# NUS Curriculum & Job Alignment Framework

This notebook implements a scalable, validated framework for assessing alignment between NUS degree programmes (constructed from `modules.csv`) and MyCareersFuture job advertisements.

**Objective:** Provide MOE with a credible, systematic framework for curriculum-job alignment analysis at the degree level.

**Sections:**
0. Setup
1. Data Loading & Preprocessing
2. Degree Profile Construction
3. Semantic Embedding
4. Baseline Cosine Similarity Retrieval
5. Job-Skill Coverage Score (Hybrid Alignment)
6. Job Clustering (Scalability Layer)
7. Evaluation Framework
8. Results Summary & Interpretation

---
## Section 0: Setup

In [ ]:
import os
import re
import json
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')

JOB_CORPUS_VERSION = 'targeted_dedup_v2'
SEMANTIC_RETRIEVAL_VERSION = 'module_top5_job120_v1'
JOB_DESC_WORD_LIMIT = 120
DEGREE_MODULE_TOP_K = 5
SKILL_SIGNAL_VERSION = 'job_skill_coverage_v1'

# ── Paths ──────────────────────────────────────────────────────────────────────
if Path.cwd().name == 'notebooks':
    REPO_ROOT = Path('..').resolve()
else:
    REPO_ROOT = Path('.').resolve()
DATA_DIR       = REPO_ROOT / 'data'
JOBS_DIR       = DATA_DIR  / 'MyCareersFutureData'
MODULES_CSV    = DATA_DIR  / 'modules.csv'
CACHE_DIR      = REPO_ROOT / 'notebooks' / 'cache'
EVAL_DIR       = REPO_ROOT / 'notebooks' / 'evaluation'

CACHE_DIR.mkdir(exist_ok=True)
EVAL_DIR.mkdir(exist_ok=True)

def listify_text_values(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        return [part.strip() for part in value.split(',') if part.strip()]
    if isinstance(value, (list, tuple, set, np.ndarray, pd.Series)):
        return [str(item).strip() for item in value if pd.notna(item) and str(item).strip()]
    return []

def truncate_words(text: str, max_words: int) -> str:
    words = str(text or '').split()
    return ' '.join(words[:max_words])

def build_structured_job_text(row: pd.Series, max_desc_words: int = JOB_DESC_WORD_LIMIT) -> str:
    parts = [str(row.get('title', '') or '').strip()]

    categories = listify_text_values(row.get('categories', []))
    skills = listify_text_values(row.get('skills', []))

    if categories:
        parts.append(f"Categories: {', '.join(categories)}.")
    if skills:
        parts.append(f"Skills: {', '.join(skills)}.")

    description = truncate_words(row.get('description_clean', ''), max_desc_words)
    if description:
        parts.append(description)

    return ' '.join(part for part in parts if part).strip()

def build_module_text(row: pd.Series) -> str:
    parts = [f"{row['moduleCode']}. {row['description_clean']}"]

    if row.get('description_skills', ''):
        parts.append(f"Skills: {row['description_skills']}.")

    return ' '.join(parts).strip()

def aggregate_top_k_similarity(score_matrix: np.ndarray, top_k: int) -> np.ndarray:
    if score_matrix.ndim != 2:
        raise ValueError('score_matrix must be a 2D array.')
    if score_matrix.shape[0] == 0:
        return np.zeros(score_matrix.shape[1], dtype=np.float32)

    k = min(top_k, score_matrix.shape[0])
    if k == score_matrix.shape[0]:
        return score_matrix.mean(axis=0, dtype=np.float32).astype(np.float32)

    top_scores = np.partition(score_matrix, score_matrix.shape[0] - k, axis=0)[-k:]
    return top_scores.mean(axis=0, dtype=np.float32).astype(np.float32)

print(f'Repo root  : {REPO_ROOT}')
print(f'Jobs dir   : {JOBS_DIR}')
print(f'Modules CSV: {MODULES_CSV}')
print(f'Cache dir  : {CACHE_DIR}')
print(f'Job corpus : {JOB_CORPUS_VERSION}')
print(f'Semantics  : {SEMANTIC_RETRIEVAL_VERSION}')


---
## Section 1: Data Loading & Preprocessing

### 1a - NUS Modules

In [ ]:
def strip_html(text: str) -> str:
    """Remove HTML tags and normalise whitespace."""
    if not isinstance(text, str) or not text.strip():
        return ''
    soup = BeautifulSoup(text, 'html.parser')
    clean = soup.get_text(separator=' ')
    return re.sub(r'\s+', ' ', clean).strip()

modules_raw = pd.read_csv(MODULES_CSV)
print(f'Raw modules shape: {modules_raw.shape}')
print(f'Columns: {list(modules_raw.columns)}')
modules_raw.head(3)

In [ ]:
def extract_level(code: str) -> int:
    """Return the numeric level of a module code (e.g. CS1010 → 1, CS5XXX → 5)."""
    digits = re.search(r'(\d)', str(code))
    return int(digits.group(1)) if digits else 9

modules = modules_raw.copy()

# Clean descriptions
modules['description_clean'] = modules['description'].apply(strip_html)
modules['module_text'] = (
    modules['title'].fillna('') + '. ' + modules['description_clean']
).str.strip('. ')

# Mark undergrad vs postgrad
modules['level'] = modules['moduleCode'].apply(extract_level)
modules['is_undergrad'] = modules['level'].between(1, 4)

# Drop rows with no usable description
modules_clean = modules[modules['description_clean'].str.len() > 20].copy()

print(f'Total modules with descriptions: {len(modules_clean):,}')
print(f'  Undergrad (level 1–4): {modules_clean["is_undergrad"].sum():,}')
print(f'  Postgrad  (level 5+) : {(~modules_clean["is_undergrad"]).sum():,}')
print(f'Faculties : {modules_clean["faculty"].nunique()}')
print(f'Departments: {modules_clean["department"].nunique()}')
modules_clean[['moduleCode','title','faculty','department','moduleCredit','is_undergrad']].head(5)

In [ ]:
# Coverage: modules with descriptions per department
dept_coverage = (
    modules_clean[modules_clean['is_undergrad']]
    .groupby('department')
    .size()
    .reset_index(name='n_modules')
    .sort_values('n_modules', ascending=False)
)

sufficient = (dept_coverage['n_modules'] >= 5).sum()
print(f'Departments with ≥5 undergrad modules: {sufficient} / {len(dept_coverage)}')
dept_coverage.head(15)

### 1b - MyCareersFuture Job Ads

We clean raw job ads, label clearly out-of-scope roles (tuition/teaching, academia, internships, very senior leadership), collapse exact reposts, then run a second-stage near-duplicate clustering pass within repeated title-company blocks before retrieval.


In [ ]:
def parse_job_file(path: Path) -> dict | None:
    """Parse a single MCF JSON file into a flat record."""
    try:
        with open(path, 'r', encoding='utf-8') as f:
            d = json.load(f)
    except Exception:
        return None

    meta = d.get('metadata', {})
    salary = d.get('salary', {})
    sal_type = (salary.get('type') or {}).get('salaryType', '')

    skills_raw = d.get('skills', []) or []
    skills_list = [s['skill'] for s in skills_raw if isinstance(s, dict) and s.get('skill')]

    cats = d.get('categories', []) or []
    categories_list = [c['category'] for c in cats if isinstance(c, dict) and c.get('category')]

    pos = d.get('positionLevels', []) or []
    position_levels = [p['position'] for p in pos if isinstance(p, dict) and p.get('position')]

    emp_types = d.get('employmentTypes', []) or []
    employment_types = [e['employmentType'] for e in emp_types if isinstance(e, dict)]

    return {
        'job_id'           : meta.get('jobPostId', path.stem),
        'title'            : d.get('title', ''),
        'description_raw'  : d.get('description', '') or '',
        'skills'           : skills_list,
        'skills_str'       : ', '.join(skills_list),
        'categories'       : categories_list,
        'categories_str'   : ', '.join(categories_list),
        'position_levels'  : position_levels,
        'employment_types' : employment_types,
        'ssoc_code'        : d.get('ssocCode', ''),
        'salary_min'       : (salary.get('minimum') or np.nan),
        'salary_max'       : (salary.get('maximum') or np.nan),
        'salary_type'      : sal_type,
        'company'          : (d.get('postedCompany') or {}).get('name', ''),
        'posted_date'      : meta.get('newPostingDate', ''),
        'expiry_date'      : meta.get('expiryDate', ''),
    }

job_files = list(JOBS_DIR.glob('*.json'))
print(f'Found {len(job_files):,} JSON job files')

In [ ]:
JOBS_CACHE = CACHE_DIR / 'jobs_raw.parquet'

if JOBS_CACHE.exists():
    print('Loading job ads from cache...')
    jobs_raw = pd.read_parquet(JOBS_CACHE)
else:
    print('Parsing job ad JSON files...')
    records = []
    for p in tqdm(job_files, desc='Loading jobs'):
        rec = parse_job_file(p)
        if rec:
            records.append(rec)
    jobs_raw = pd.DataFrame(records)
    jobs_raw.to_parquet(JOBS_CACHE, index=False)
    print(f'Cached to {JOBS_CACHE}')

print(f'Jobs loaded: {len(jobs_raw):,}')
jobs_raw.head(3)

In [ ]:
# Clean, filter, and deduplicate job ads before retrieval
import ast
import hashlib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

LIST_COLUMNS = ['skills', 'categories', 'position_levels', 'employment_types']
SEMANTIC_DEDUP_SIM_THRESHOLD = 0.985
MAX_SEMANTIC_BLOCK_SIZE = 40


def ensure_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple, set, np.ndarray)):
        return [item for item in value if pd.notna(item)]
    if pd.isna(value):
        return []
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except (SyntaxError, ValueError):
            pass
        return [part.strip() for part in value.split(',') if part.strip()]
    return []



def normalise_job_field(text: str) -> str:
    text = strip_html(str(text or ''))
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()



def classify_role_scope(row: pd.Series) -> str:
    title = normalise_job_field(row.get('title', ''))
    company = normalise_job_field(row.get('company', ''))
    categories = normalise_job_field(' '.join(row.get('categories', []) or []))
    position_levels = normalise_job_field(' '.join(row.get('position_levels', []) or []))
    employment_types = normalise_job_field(' '.join(row.get('employment_types', []) or []))
    scope_text = ' '.join([title, company, categories, position_levels, employment_types])

    if re.search(r'\b(intern(ship)?|industrial attachment|student assistant)\b', scope_text):
        return 'exclude_internship'
    if re.search(r'\b(professor|postdoctoral?|research fellow|academic|dean|faculty)\b', scope_text):
        return 'exclude_academia'
    if re.search(r'\b(tuition|tutor|teacher|lecturer|instructor)\b', scope_text):
        return 'exclude_tuition_teaching'
    if (
        re.search(r'\b(chief|director|vice president|president|head of|managing director|partner)\b', scope_text)
        or 'senior management' in position_levels
    ):
        return 'exclude_very_senior'
    return 'in_scope'



def build_job_fingerprint(row: pd.Series) -> str:
    title_norm = normalise_job_field(row.get('title', ''))
    company_norm = normalise_job_field(row.get('company', '')) or normalise_job_field(row.get('ssoc_code', ''))
    desc_norm = normalise_job_field(row.get('description_clean', ''))
    payload = ' || '.join([title_norm, company_norm, desc_norm])
    return hashlib.sha1(payload.encode('utf-8')).hexdigest()



def cluster_semantic_duplicates(group: pd.DataFrame) -> pd.DataFrame:
    group = group.copy()
    n_rows = len(group)

    group['semantic_cluster_local_id'] = np.arange(n_rows)
    group['semantic_cluster_size'] = 1
    group['semantic_similarity_max'] = np.nan
    group['semantic_block_skipped'] = False

    if n_rows <= 1:
        return group

    if n_rows > MAX_SEMANTIC_BLOCK_SIZE:
        group['semantic_block_skipped'] = True
        return group

    cluster_text = (
        group['description_clean'].fillna('')
        + ' Skills: '
        + group['skills_str'].fillna('')
        + ' Categories: '
        + group['categories_str'].fillna('')
    ).map(normalise_job_field)

    if cluster_text.str.len().eq(0).all():
        group['semantic_block_skipped'] = True
        return group

    matrix = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=1).fit_transform(cluster_text)
    sims = cosine_similarity(matrix)

    parent = list(range(n_rows))

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a: int, b: int) -> None:
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n_rows):
        for j in range(i + 1, n_rows):
            if sims[i, j] >= SEMANTIC_DEDUP_SIM_THRESHOLD:
                union(i, j)

    cluster_map = {}
    cluster_sizes = {}
    cluster_similarity = {}
    next_cluster_id = 0

    for i in range(n_rows):
        root = find(i)
        if root not in cluster_map:
            cluster_map[root] = next_cluster_id
            members = [m for m in range(n_rows) if find(m) == root]
            cluster_sizes[root] = len(members)
            if len(members) > 1:
                member_sims = sims[np.ix_(members, members)]
                upper = member_sims[np.triu_indices(len(members), k=1)]
                cluster_similarity[root] = float(upper.max()) if len(upper) else 1.0
            else:
                cluster_similarity[root] = np.nan
            next_cluster_id += 1

        group.iloc[i, group.columns.get_loc('semantic_cluster_local_id')] = cluster_map[root]
        group.iloc[i, group.columns.get_loc('semantic_cluster_size')] = cluster_sizes[root]
        group.iloc[i, group.columns.get_loc('semantic_similarity_max')] = cluster_similarity[root]

    return group


jobs_candidates = jobs_raw.copy()
jobs_candidates['description_clean'] = jobs_candidates['description_raw'].apply(strip_html)

before = len(jobs_candidates)
jobs_candidates = jobs_candidates[
    jobs_candidates['description_clean'].str.len() > 30
].drop_duplicates('job_id').copy()

for col in LIST_COLUMNS:
    if col in jobs_candidates.columns:
        jobs_candidates[col] = jobs_candidates[col].apply(ensure_list)

jobs_candidates['job_text'] = jobs_candidates.apply(build_structured_job_text, axis=1)

jobs_candidates['title_norm'] = jobs_candidates['title'].apply(normalise_job_field)
jobs_candidates['company_norm'] = jobs_candidates['company'].apply(normalise_job_field)
jobs_candidates['ssoc_norm'] = jobs_candidates['ssoc_code'].astype(str).apply(normalise_job_field)

jobs_candidates['role_scope'] = jobs_candidates.apply(classify_role_scope, axis=1)
jobs_candidates['is_target_role'] = jobs_candidates['role_scope'].eq('in_scope')
jobs_candidates['job_fingerprint'] = jobs_candidates.apply(build_job_fingerprint, axis=1)
jobs_candidates['posted_date_dt'] = pd.to_datetime(jobs_candidates['posted_date'], errors='coerce')

salary_max_num = pd.to_numeric(jobs_candidates['salary_max'], errors='coerce')
salary_min_num = pd.to_numeric(jobs_candidates['salary_min'], errors='coerce')
jobs_candidates['salary_sort'] = salary_max_num.fillna(salary_min_num).fillna(-1)

jobs_candidates = jobs_candidates.sort_values(
    ['job_fingerprint', 'posted_date_dt', 'salary_sort', 'job_id'],
    ascending=[True, False, False, True]
).copy()

jobs_candidates['duplicate_count'] = jobs_candidates.groupby('job_fingerprint')['job_id'].transform('size')
jobs_candidates['duplicate_rank'] = jobs_candidates.groupby('job_fingerprint').cumcount() + 1
jobs_candidates['is_near_duplicate'] = jobs_candidates['duplicate_count'].gt(1)
jobs_candidates['canonical_job_id'] = jobs_candidates.groupby('job_fingerprint')['job_id'].transform('first')

jobs_scope_audit = jobs_candidates.drop(columns=['posted_date_dt', 'salary_sort'], errors='ignore').copy()
jobs_excluded = jobs_scope_audit[~jobs_scope_audit['is_target_role']].copy()
jobs_exact_removed = jobs_scope_audit[
    jobs_scope_audit['is_target_role'] & jobs_scope_audit['duplicate_rank'].gt(1)
].copy()

jobs_for_semantic_cluster = jobs_scope_audit[
    jobs_scope_audit['is_target_role'] & jobs_scope_audit['duplicate_rank'].eq(1)
].copy()

jobs_for_semantic_cluster['semantic_block_key'] = (
    jobs_for_semantic_cluster['title_norm']
    + ' || '
    + jobs_for_semantic_cluster['company_norm'].where(
        jobs_for_semantic_cluster['company_norm'].str.len() > 0,
        jobs_for_semantic_cluster['ssoc_norm']
    )
)

semantic_cluster_frames = []
for _, group in jobs_for_semantic_cluster.groupby('semantic_block_key', sort=False):
    semantic_cluster_frames.append(cluster_semantic_duplicates(group))

if semantic_cluster_frames:
    jobs_for_semantic_cluster = pd.concat(semantic_cluster_frames, ignore_index=True)
else:
    jobs_for_semantic_cluster = jobs_for_semantic_cluster.copy()

jobs_for_semantic_cluster['semantic_group_key'] = (
    jobs_for_semantic_cluster['semantic_block_key']
    + ' ## '
    + jobs_for_semantic_cluster['semantic_cluster_local_id'].astype(str)
)

jobs_for_semantic_cluster = jobs_for_semantic_cluster.sort_values(
    ['semantic_group_key', 'posted_date', 'job_id'],
    ascending=[True, False, True]
).copy()

jobs_for_semantic_cluster['semantic_duplicate_count'] = jobs_for_semantic_cluster.groupby('semantic_group_key')['job_id'].transform('size')
jobs_for_semantic_cluster['semantic_duplicate_rank'] = jobs_for_semantic_cluster.groupby('semantic_group_key').cumcount() + 1
jobs_for_semantic_cluster['canonical_semantic_job_id'] = jobs_for_semantic_cluster.groupby('semantic_group_key')['job_id'].transform('first')
jobs_for_semantic_cluster['is_semantic_near_duplicate'] = jobs_for_semantic_cluster['semantic_duplicate_count'].gt(1)

semantic_cols = [
    'job_id',
    'semantic_block_key',
    'semantic_cluster_local_id',
    'semantic_group_key',
    'semantic_cluster_size',
    'semantic_similarity_max',
    'semantic_block_skipped',
    'semantic_duplicate_count',
    'semantic_duplicate_rank',
    'canonical_semantic_job_id',
    'is_semantic_near_duplicate',
]

jobs_scope_audit = jobs_scope_audit.merge(
    jobs_for_semantic_cluster[semantic_cols],
    on='job_id',
    how='left',
    validate='one_to_one'
)

jobs_semantic_removed = jobs_for_semantic_cluster[
    jobs_for_semantic_cluster['semantic_duplicate_rank'].gt(1)
].copy()

jobs = jobs_for_semantic_cluster[
    jobs_for_semantic_cluster['semantic_duplicate_rank'].eq(1)
].copy()

jobs = jobs.sort_values(['posted_date', 'job_id'], ascending=[False, True]).reset_index(drop=True)

scope_counts = jobs_excluded['role_scope'].value_counts().sort_values(ascending=False)
exact_duplicate_groups = jobs_exact_removed['job_fingerprint'].nunique()
semantic_duplicate_groups = jobs_semantic_removed['semantic_group_key'].nunique()

print(f'Jobs after description filter & job_id dedup: {len(jobs_scope_audit):,} (dropped {before - len(jobs_scope_audit):,})')
print(f'Out-of-scope jobs removed: {len(jobs_excluded):,}')
if not scope_counts.empty:
    print(scope_counts.to_string())
print(f'In-scope jobs before exact dedup: {jobs_scope_audit["is_target_role"].sum():,}')
print(f'Exact repost rows removed: {len(jobs_exact_removed):,} across {exact_duplicate_groups:,} groups')
print(f'Jobs after exact dedup and before semantic clustering: {len(jobs_for_semantic_cluster):,}')
print(f'Semantic near-duplicate rows removed: {len(jobs_semantic_removed):,} across {semantic_duplicate_groups:,} groups')
print(f'Final in-scope jobs used for retrieval: {len(jobs):,}')

print(f'Jobs with at least one skill tag: {jobs["skills"].apply(len).gt(0).sum():,}')
print(f'Jobs with descriptions capped to first {JOB_DESC_WORD_LIMIT} words for embeddings: {jobs["description_clean"].str.split().str.len().gt(JOB_DESC_WORD_LIMIT).sum():,}')
print(f'Unique categories in final corpus: {set(c for cats in jobs["categories"] for c in cats)}')


In [ ]:
JOB_META_CACHE = CACHE_DIR / f'jobs_clean_{JOB_CORPUS_VERSION}.parquet'
JOB_SCOPE_CACHE = CACHE_DIR / f'jobs_scope_audit_{JOB_CORPUS_VERSION}.parquet'

jobs.to_parquet(JOB_META_CACHE, index=False)
jobs_scope_audit.to_parquet(JOB_SCOPE_CACHE, index=False)

print(f'Saved filtered in-scope jobs to: {JOB_META_CACHE}')
print(f'Saved job scope audit table to: {JOB_SCOPE_CACHE}')


In [ ]:
# Parsing quality check
total_files = len(job_files)
parsed_jobs = len(jobs_raw)
clean_jobs = len(jobs_scope_audit)
out_of_scope_jobs = len(jobs_excluded)
in_scope_before_exact = int(jobs_scope_audit['is_target_role'].sum())
exact_removed_jobs = len(jobs_exact_removed)
post_exact_jobs = len(jobs_for_semantic_cluster)
semantic_removed_jobs = len(jobs_semantic_removed)
final_jobs = len(jobs)

print('=== Job Ad Coverage ===')
print(f'  JSON files found                 : {total_files:>7,}')
print(f'  Successfully parsed              : {parsed_jobs:>7,}  ({parsed_jobs/total_files*100:.1f}%)')
print(f'  Clean after text filter          : {clean_jobs:>7,}  ({clean_jobs/total_files*100:.1f}%)')
print(f'  Excluded as out-of-scope         : {out_of_scope_jobs:>7,}  ({out_of_scope_jobs/total_files*100:.1f}%)')
print(f'  In-scope before exact dedup      : {in_scope_before_exact:>7,}  ({in_scope_before_exact/total_files*100:.1f}%)')
print(f'  Removed as exact repost          : {exact_removed_jobs:>7,}  ({exact_removed_jobs/total_files*100:.1f}%)')
print(f'  After exact dedup                : {post_exact_jobs:>7,}  ({post_exact_jobs/total_files*100:.1f}%)')
print(f'  Removed as semantic near-dup     : {semantic_removed_jobs:>7,}  ({semantic_removed_jobs/total_files*100:.1f}%)')
print(f'  Final jobs used for retrieval    : {final_jobs:>7,}  ({final_jobs/total_files*100:.1f}%)')

if not jobs_excluded.empty:
    print('\n  Out-of-scope breakdown:')
    for reason, count in jobs_excluded['role_scope'].value_counts().items():
        print(f'    - {reason:<26} {count:>7,}')

print('\n=== Module Coverage ===')
total_mods = len(modules_raw)
usable_mods = len(modules_clean)
ug_mods = modules_clean['is_undergrad'].sum()
print(f'  Total module rows                : {total_mods:>7,}')
print(f'  With usable desc                 : {usable_mods:>7,}  ({usable_mods/total_mods*100:.1f}%)')
print(f'  Undergrad (1–4xxx)               : {ug_mods:>7,}  ({ug_mods/total_mods*100:.1f}%)')


---
## Section 2: Degree Profile Construction

We build baskets of modules for dsa, cs, cnm, ce and biz. For each degree, they have 15 core modules and 8 common curriculum modules. We then concatenate the module texts to module code.

In [ ]:
DEGREE_MAP_CSV = DATA_DIR / 'degree_to_module_map.csv'
MODULE_SKILLS_CSV = DATA_DIR / 'nus_modules_skills_output.csv'

degree_map = pd.read_csv(DEGREE_MAP_CSV)
module_skills = pd.read_csv(MODULE_SKILLS_CSV)

# Clean headers
degree_map.columns = degree_map.columns.str.strip()
module_skills.columns = module_skills.columns.str.strip()

# Normalise join keys
degree_map['moduleCode'] = degree_map['moduleCode'].astype(str).str.strip().str.upper()
module_skills['moduleCode'] = module_skills['moduleCode'].astype(str).str.strip().str.upper()

# Keep only the columns we need from the skills file
module_skill_cols = [
    'moduleCode',
    'title',
    'description_clean',
    'description_skills',
    'top_skills',
]

degree_modules = degree_map.merge(
    module_skills[module_skill_cols],
    on='moduleCode',
    how='left',
    validate='many_to_one'
)

# Clean merged text fields
for col in ['title', 'description_clean', 'description_skills', 'top_skills']:
    degree_modules[col] = degree_modules[col].fillna('').astype(str).str.strip()

degree_modules['requirement_group'] = pd.Categorical(
    degree_modules['requirement_group'].str.strip().str.lower(),
    categories=['core', 'common'],
    ordered=True
)

print(f'degree_map rows      : {len(degree_map):,}')
print(f'merged rows          : {len(degree_modules):,}')
print(f'missing module joins : {degree_modules['title'].eq('').sum():,}')

display(
    degree_modules[
        ['degree_id', 'degree_name', 'requirement_group', 'moduleCode', 'title', 'top_skills']
    ].head(10)
)

In [ ]:
MAX_WORDS_PER_PROFILE = 8000

degree_modules = degree_modules.copy()
degree_modules['module_order'] = np.arange(len(degree_modules))

# Keep only rows with usable text
degree_modules = degree_modules[
    degree_modules['description_clean'].str.len() > 0
].copy()

def build_module_text(row: pd.Series) -> str:
    """Create one text block per module using code + description + extracted skills."""
    parts = [f"{row['moduleCode']}. {row['description_clean']}"]

    if row['description_skills']:
        parts.append(f"Skills: {row['description_skills']}.")

    return ' '.join(parts).strip()

degree_modules['module_profile_text'] = degree_modules.apply(build_module_text, axis=1)

def build_profile_text(group: pd.DataFrame, max_words: int = MAX_WORDS_PER_PROFILE) -> str:
    group = group.sort_values(['requirement_group', 'module_order'])

    result = []
    word_count = 0

    for text in group['module_profile_text']:
        words = text.split()
        if word_count + len(words) > max_words:
            remaining = max_words - word_count
            if remaining > 10:
                result.append(' '.join(words[:remaining]))
            break
        result.append(text)
        word_count += len(words)

    return ' '.join(result)

degree_profiles = (
    degree_modules
    .groupby(['degree_id', 'degree_name'], as_index=False)
    .apply(lambda g: pd.Series({
        'profile_text': build_profile_text(g),
        'n_modules': len(g),
        'word_count': len(build_profile_text(g).split())
    }))
    .reset_index(drop=True)
)

degree_profiles['degree_label'] = degree_profiles['degree_name']

print(f'Built {len(degree_profiles)} degree profiles')
display(degree_profiles[['degree_id', 'degree_name', 'n_modules', 'word_count']])

In [ ]:
# Sample profile + module basket
sample_degree_id = 'cs'   # change to: dsa, cs, cnm, ce, biz

sample_profile = degree_profiles[degree_profiles['degree_id'] == sample_degree_id]
sample_modules = degree_modules[degree_modules['degree_id'] == sample_degree_id].copy()

if sample_profile.empty:
    print(f'No profile found for degree_id={sample_degree_id!r}')
else:
    row = sample_profile.iloc[0]
    print(f"Degree: {row['degree_name']} ({row['degree_id']})")
    print(f"Modules used: {row['n_modules']}")
    print(f"Word count: {row['word_count']}")

    print('\n--- Module basket ---')
    display(
        sample_modules[
            ['requirement_group', 'moduleCode', 'title', 'top_skills']
        ].sort_values(['requirement_group', 'moduleCode'])
    )

    print('\n--- Profile preview (first 2000 chars) ---\n')
    print(row['profile_text'][:2000])

In [ ]:
DEGREE_META_CACHE = CACHE_DIR / "degree_profiles.parquet"
DEGREE_MODULES_CACHE = CACHE_DIR / "degree_modules.parquet"

degree_profiles.to_parquet(DEGREE_META_CACHE, index=False)
degree_modules.to_parquet(DEGREE_MODULES_CACHE, index=False)
print(f"Saved degree profiles to: {DEGREE_META_CACHE}")
print(f"Saved degree modules to: {DEGREE_MODULES_CACHE}")

---
## Section 3: Semantic Embedding

We encode all degree profiles and job descriptions using a pre-trained sentence transformer. Embeddings are cached to disk so re-runs are fast.

In [ ]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'all-MiniLM-L6-v2'  # swap for all-mpnet-base-v2 for higher quality

print(f'Loading model: {MODEL_NAME}')
try:
    model = SentenceTransformer(MODEL_NAME, local_files_only=True)
    print('Model loaded from local cache.')
except Exception:
    print('Local model cache not found; attempting to download...')
    model = SentenceTransformer(MODEL_NAME)
    print('Model downloaded and loaded.')

In [ ]:
DEGREE_META_CACHE = CACHE_DIR / 'degree_profiles.parquet'
DEGREE_MODULES_CACHE = CACHE_DIR / 'degree_modules.parquet'
DEGREE_MODULE_EMB_CACHE = CACHE_DIR / f'degree_module_embeddings_{MODEL_NAME.replace("/","_")}_{SEMANTIC_RETRIEVAL_VERSION}.npy'
DEGREE_EMB_CACHE = CACHE_DIR / f'degree_embeddings_{MODEL_NAME.replace("/","_")}_{SEMANTIC_RETRIEVAL_VERSION}.npy'

# Section 3 depends on degree_profiles and degree_modules from Section 2.
# If you restarted the notebook, load the Section 2 outputs from local cache.
if 'degree_profiles' not in globals():
    if DEGREE_META_CACHE.exists():
        degree_profiles = pd.read_parquet(DEGREE_META_CACHE)
        print(f'Loaded degree profiles from: {DEGREE_META_CACHE}')
    else:
        raise FileNotFoundError('Run Section 2 first to build degree_profiles.')

if 'degree_modules' not in globals():
    if DEGREE_MODULES_CACHE.exists():
        degree_modules = pd.read_parquet(DEGREE_MODULES_CACHE)
        print(f'Loaded degree modules from: {DEGREE_MODULES_CACHE}')
    else:
        raise FileNotFoundError('Run Section 2 first to build degree_modules.')

degree_profiles = degree_profiles.copy()
degree_profiles['degree_id'] = degree_profiles['degree_id'].astype(str)
degree_modules = degree_modules.copy()
degree_modules['degree_id'] = degree_modules['degree_id'].astype(str)

if 'module_profile_text' not in degree_modules.columns:
    required_cols = {'moduleCode', 'description_clean', 'description_skills'}
    if required_cols.issubset(degree_modules.columns):
        degree_modules['module_profile_text'] = degree_modules.apply(build_module_text, axis=1)
    else:
        raise ValueError('Cached degree_modules is missing module_profile_text. Re-run Section 2.')

rebuild_module_embeddings = True
if DEGREE_MODULE_EMB_CACHE.exists():
    cached_module_embeddings = np.load(DEGREE_MODULE_EMB_CACHE)
    if cached_module_embeddings.ndim == 2 and cached_module_embeddings.shape[0] == len(degree_modules):
        print('Loading degree-module embeddings from cache...')
        module_embeddings = cached_module_embeddings
        rebuild_module_embeddings = False
    else:
        print('Cached degree-module embeddings do not match the current degree_modules table; rebuilding...')

if rebuild_module_embeddings:
    print(f'Encoding {len(degree_modules)} degree-module texts...')
    t0 = time.time()
    module_embeddings = model.encode(
        degree_modules['module_profile_text'].tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    elapsed = time.time() - t0
    np.save(DEGREE_MODULE_EMB_CACHE, module_embeddings)
    degree_modules.to_parquet(DEGREE_MODULES_CACHE, index=False)
    print(f'Done in {elapsed:.1f}s - saved to {DEGREE_MODULE_EMB_CACHE}')

degree_module_ids = degree_modules['degree_id'].to_numpy()
degree_module_indices = {
    degree_id: np.flatnonzero(degree_module_ids == degree_id)
    for degree_id in degree_profiles['degree_id'].tolist()
}

rebuild_degree_embeddings = rebuild_module_embeddings
if DEGREE_EMB_CACHE.exists() and not rebuild_module_embeddings:
    cached_degree_embeddings = np.load(DEGREE_EMB_CACHE)
    if cached_degree_embeddings.ndim == 2 and cached_degree_embeddings.shape[0] == len(degree_profiles):
        print('Loading aggregated degree embeddings from cache...')
        degree_embeddings = cached_degree_embeddings
        rebuild_degree_embeddings = False
    else:
        print('Cached aggregated degree embeddings do not match the current degree_profiles; rebuilding...')
elif rebuild_module_embeddings:
    print('Module embeddings changed; rebuilding aggregated degree embeddings...')

if rebuild_degree_embeddings:
    print(f'Aggregating {len(degree_profiles)} degree embeddings from module embeddings...')
    degree_embeddings = np.zeros((len(degree_profiles), module_embeddings.shape[1]), dtype=np.float32)

    for d_idx, degree_id in enumerate(degree_profiles['degree_id'].tolist()):
        module_idx = degree_module_indices.get(degree_id, np.array([], dtype=int))
        if len(module_idx) == 0:
            continue

        centroid = module_embeddings[module_idx].mean(axis=0)
        norm = np.linalg.norm(centroid)
        degree_embeddings[d_idx] = centroid / norm if norm > 0 else centroid

    np.save(DEGREE_EMB_CACHE, degree_embeddings)
    degree_profiles.to_parquet(DEGREE_META_CACHE, index=False)
    print(f'Saved aggregated degree embeddings to: {DEGREE_EMB_CACHE}')

print(f'Degree-module embeddings shape: {module_embeddings.shape}')
print(f'Aggregated degree embeddings shape: {degree_embeddings.shape}')

In [ ]:
# takes about 3min to run
JOB_EMB_CACHE = CACHE_DIR / f'job_embeddings_{MODEL_NAME.replace("/","_")}_{JOB_CORPUS_VERSION}_{SEMANTIC_RETRIEVAL_VERSION}.npy'
JOB_META_CACHE = CACHE_DIR / f'jobs_clean_{JOB_CORPUS_VERSION}.parquet'

if 'jobs' not in globals():
    if JOB_META_CACHE.exists():
        jobs = pd.read_parquet(JOB_META_CACHE)
        print(f'Loaded clean jobs from: {JOB_META_CACHE}')
    else:
        raise FileNotFoundError('Run Section 1 first to build the filtered in-scope jobs corpus.')

jobs = jobs.copy()
jobs['job_text'] = jobs.apply(build_structured_job_text, axis=1)

rebuild_job_embeddings = True
if JOB_EMB_CACHE.exists():
    cached_job_embeddings = np.load(JOB_EMB_CACHE)
    if cached_job_embeddings.ndim == 2 and cached_job_embeddings.shape[0] == len(jobs):
        print('Loading job embeddings from cache...')
        job_embeddings = cached_job_embeddings
        rebuild_job_embeddings = False
    else:
        print('Cached job embeddings do not match the current filtered job corpus; rebuilding...')

if rebuild_job_embeddings:
    print(f'Encoding {len(jobs):,} job descriptions...')
    t0 = time.time()
    job_embeddings = model.encode(
        jobs['job_text'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    elapsed = time.time() - t0
    np.save(JOB_EMB_CACHE, job_embeddings)
    jobs.to_parquet(JOB_META_CACHE, index=False)
    print(f'Done in {elapsed:.1f}s - saved to {JOB_EMB_CACHE}')

print(f'Job embeddings shape: {job_embeddings.shape}')


---
## Section 4: Baseline Cosine Similarity Retrieval

Since embeddings are unit-normalised, cosine similarity equals the dot product. We compute a full **degrees × jobs** similarity matrix and retrieve top-K results in both directions.

In [ ]:
import numpy as np

SIM_MATRIX_CACHE = CACHE_DIR / f'sim_matrix_{MODEL_NAME.replace("/","_")}_{JOB_CORPUS_VERSION}_{SEMANTIC_RETRIEVAL_VERSION}.npy'

# Full similarity matrix: (n_degrees × n_jobs)
if module_embeddings.shape[0] != len(degree_modules):
    raise ValueError('module_embeddings does not match current degree_modules. Re-run Section 3.')

if job_embeddings.shape[0] != len(jobs):
    raise ValueError('job_embeddings does not match the current filtered job corpus. Re-run Section 3.')

print(f'Computing degree-job similarity matrix using mean top-{DEGREE_MODULE_TOP_K} module scores...')
t0 = time.time()
module_job_sim = module_embeddings @ job_embeddings.T  # shape: (M, J)
sim_matrix = np.zeros((len(degree_profiles), len(jobs)), dtype=np.float32)

for d_idx, degree_id in enumerate(degree_profiles['degree_id'].tolist()):
    module_idx = degree_module_indices.get(degree_id, np.array([], dtype=int))
    sim_matrix[d_idx] = aggregate_top_k_similarity(module_job_sim[module_idx], DEGREE_MODULE_TOP_K)

np.save(SIM_MATRIX_CACHE, sim_matrix)
print(f'Similarity matrix shape: {sim_matrix.shape}  [{time.time()-t0:.1f}s]')
print(f'Saved similarity matrix to: {SIM_MATRIX_CACHE}')


In [ ]:
K = 10

def top_k_jobs_for_degree(degree_idx: int, k: int = K) -> pd.DataFrame:
    """Return the top-k job ads most similar to a given degree profile."""
    scores = sim_matrix[degree_idx]
    top_idx = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({
        'rank'       : range(1, k+1),
        'job_id'     : jobs.iloc[top_idx]['job_id'].values,
        'job_title'  : jobs.iloc[top_idx]['title'].values,
        'categories' : jobs.iloc[top_idx]['categories_str'].values,
        'sim_score'  : scores[top_idx].round(4),
    })

def top_k_degrees_for_job(job_idx: int, k: int = K) -> pd.DataFrame:
    """Return the top-k degree profiles most similar to a given job ad."""
    scores = sim_matrix[:, job_idx]
    top_idx = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({
        'rank'       : range(1, k + 1),
        'degree_id'  : degree_profiles.iloc[top_idx]['degree_id'].values,
        'degree'     : degree_profiles.iloc[top_idx]['degree_name'].values,
        'sim_score'  : scores[top_idx].round(4),
    })

In [ ]:
# Demo: Top-10 jobs for Computer Science
cs_idx = degree_profiles[degree_profiles['degree_id'] == 'cs'].index

if len(cs_idx) > 0:
    idx = degree_profiles.index.get_loc(cs_idx[0])
    degree_name = degree_profiles.iloc[idx]['degree_name']
    print(f'Top-{K} jobs for: {degree_name}')
    display(top_k_jobs_for_degree(idx, K))
else:
    print('No CS degree found — showing first degree profile instead.')
    display(top_k_jobs_for_degree(0, K))

In [ ]:
# Demo: Top-10 jobs for Data Science and Analytics
dsa_idx = degree_profiles[degree_profiles['degree_id'] == 'dsa'].index

if len(dsa_idx) > 0:
    idx = degree_profiles.index.get_loc(dsa_idx[0])
    degree_name = degree_profiles.iloc[idx]['degree_name']
    print(f'Top-{K} jobs for: {degree_name}')
    display(top_k_jobs_for_degree(idx, K))
else:
    print('No DSA degree found — showing first degree profile instead.')
    display(top_k_jobs_for_degree(0, K))

In [ ]:
# Distribution of max similarity scores per degree (health check)
import matplotlib.pyplot as plt

max_sim_per_degree = sim_matrix.max(axis=1)
mean_sim_per_degree = sim_matrix.mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(max_sim_per_degree, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Max Cosine Similarity\n(best job match per degree)')
axes[0].set_xlabel('Max Similarity Score')
axes[0].set_ylabel('Number of Degrees')

axes[1].hist(sim_matrix.max(axis=0), bins=40, color='darkorange', edgecolor='white')
axes[1].set_title('Distribution of Max Cosine Similarity\n(best degree match per job)')
axes[1].set_xlabel('Max Similarity Score')
axes[1].set_ylabel('Number of Jobs')

plt.tight_layout()
plt.savefig(CACHE_DIR / 'similarity_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Degrees where best match > 0.5: {(max_sim_per_degree > 0.5).sum()} / {len(degree_profiles)}')

---
## Section 5: Job-Skill Coverage Score (Hybrid Alignment)

We complement semantic similarity with an explicit job-skill coverage signal. The structured `skills[]` field in job ads provides a clean skill vocabulary. We match this against degree profile text lexically and score how much of each job's tagged skill set is covered.

In [ ]:
# Build a global skill vocabulary from all job ads
all_skills = set()
for skill_list in jobs['skills']:
    if isinstance(skill_list, list):
        all_skills.update(s.lower().strip() for s in skill_list if s)

print(f'Unique skills in job ads: {len(all_skills):,}')

# Sort by length descending to match longer phrases first
skill_vocab = sorted(all_skills, key=len, reverse=True)
print('Sample skills:', skill_vocab[:20])

In [ ]:
def extract_skills_from_text(text: str, vocab: list[str]) -> set[str]:
    """Find which skills from vocab appear (case-insensitive, whole-word)."""
    text_lower = str(text).lower()
    found = set()

    for skill in vocab:
        pattern = r'\b' + re.escape(skill.lower()) + r'\b'
        if re.search(pattern, text_lower):
            found.add(skill)

    return found


def job_skill_coverage(degree_skills: set, job_skills: set) -> float:
    if not job_skills:
        return 0.0
    return len(degree_skills & job_skills) / len(job_skills)


print('Extracting skills from degree profiles...')
DEGREE_SKILLS_CACHE = CACHE_DIR / 'degree_skills_actual_degrees.json'

expected_ids = set(degree_profiles['degree_id'].astype(str))
degree_skills = {}
rebuild_cache = True

if DEGREE_SKILLS_CACHE.exists():
    with open(DEGREE_SKILLS_CACHE, 'r') as f:
        degree_skills_raw = json.load(f)

    cached_keys = set(degree_skills_raw.keys())

    if cached_keys == expected_ids:
        degree_skills = {k: set(v) for k, v in degree_skills_raw.items()}
        rebuild_cache = False
        print('Loaded from cache.')
    else:
        print('Cache keys do not match current degree_ids; rebuilding cache...')

if rebuild_cache:
    degree_skills = {}
    for _, row in tqdm(
        degree_profiles.iterrows(),
        total=len(degree_profiles),
        desc='Degree skill extraction'
    ):
        degree_id = str(row['degree_id'])
        degree_skills[degree_id] = extract_skills_from_text(row['profile_text'], skill_vocab)

    with open(DEGREE_SKILLS_CACHE, 'w') as f:
        json.dump({k: sorted(v) for k, v in degree_skills.items()}, f, indent=2)

    print(f'Cached degree skills to: {DEGREE_SKILLS_CACHE}')

# Show skills found for CS
cs_row = degree_profiles[degree_profiles['degree_id'] == 'cs']

if not cs_row.empty:
    degree_id = cs_row.iloc[0]['degree_id']
    degree_name = cs_row.iloc[0]['degree_name']
    print(f'\nSkills found in {degree_name} ({degree_id}):')
    print(sorted(degree_skills.get(degree_id, set()))[:30])
else:
    print('CS degree not found in degree_profiles.')

In [ ]:
print(list(degree_skills.keys())[:10])

In [ ]:
SKILL_COVERAGE_CACHE = CACHE_DIR / f'skill_coverage_matrix_{JOB_CORPUS_VERSION}_{SKILL_SIGNAL_VERSION}.npy'
print(f'Skill coverage cache: {SKILL_COVERAGE_CACHE}')


In [ ]:
# Job-skill coverage for each job's skill set against each degree's extracted skills
# Shape: (n_degrees × n_jobs)
SKILL_COVERAGE_CACHE = CACHE_DIR / f'skill_coverage_matrix_{JOB_CORPUS_VERSION}_{SKILL_SIGNAL_VERSION}.npy'
expected_shape = (len(degree_profiles), len(jobs))

rebuild_skill_coverage = True
if SKILL_COVERAGE_CACHE.exists():
    cached_skill_coverage = np.load(SKILL_COVERAGE_CACHE)
    if cached_skill_coverage.shape == expected_shape:
        print('Loading job-skill coverage matrix from cache...')
        skill_coverage_matrix = cached_skill_coverage
        rebuild_skill_coverage = False
    else:
        print('Cached skill coverage matrix does not match the current degree/job corpus; rebuilding...')

if rebuild_skill_coverage:
    print('Computing job-skill coverage matrix...')
    n_deg = len(degree_profiles)
    n_job = len(jobs)
    skill_coverage_matrix = np.zeros((n_deg, n_job), dtype=np.float32)

    # Pre-convert job skills to list-of-sets for speed
    job_skill_sets = [
        set(s.lower().strip() for s in row if s)
        for row in jobs['skills'].tolist()
    ]

    for d_idx, (_, d_row) in enumerate(tqdm(degree_profiles.iterrows(), total=n_deg, desc='Job-skill coverage')):
        d_skills = degree_skills.get(d_row['degree_id'], set())
        if not d_skills:
            continue
        for j_idx, j_skills in enumerate(job_skill_sets):
            skill_coverage_matrix[d_idx, j_idx] = job_skill_coverage(d_skills, j_skills)

    np.save(SKILL_COVERAGE_CACHE, skill_coverage_matrix)
    print(f'Saved skill coverage matrix → {SKILL_COVERAGE_CACHE}')

print(f'Skill coverage matrix shape: {skill_coverage_matrix.shape}')
print(f'Non-zero entries: {(skill_coverage_matrix > 0).sum():,}')


In [ ]:
# Hybrid alignment score: weighted combination
ALPHA = 0.7  # weight for semantic similarity
BETA  = 0.3  # weight for job-skill coverage

if sim_matrix.shape != skill_coverage_matrix.shape:
    raise ValueError(
        f'sim_matrix shape {sim_matrix.shape} does not match skill_coverage_matrix shape {skill_coverage_matrix.shape}. '
        'Re-run Sections 3-5 so both matrices are rebuilt from the same degree/job corpus.'
    )

hybrid_matrix = ALPHA * sim_matrix + BETA * skill_coverage_matrix
print(f'Hybrid score matrix shape: {hybrid_matrix.shape}')
print(f'Score range: [{hybrid_matrix.min():.4f}, {hybrid_matrix.max():.4f}]')

def top_k_jobs_hybrid(degree_idx: int, k: int = K) -> pd.DataFrame:
    """Return top-k jobs by hybrid alignment score for a given degree."""
    scores = hybrid_matrix[degree_idx]
    top_idx = np.argsort(scores)[::-1][:k]
    return pd.DataFrame({
        'rank'        : range(1, k+1),
        'job_id'      : jobs.iloc[top_idx]['job_id'].values,
        'job_title'   : jobs.iloc[top_idx]['title'].values,
        'categories'  : jobs.iloc[top_idx]['categories_str'].values,
        'cosine_sim'  : sim_matrix[degree_idx, top_idx].round(4),
        'skill_coverage': skill_coverage_matrix[degree_idx, top_idx].round(4),
        'hybrid_score': scores[top_idx].round(4),
    })

In [ ]:
# Compare baseline vs hybrid for Computer Science
cs_idx = degree_profiles[degree_profiles['degree_id'] == 'cs'].index

if len(cs_idx) > 0:
    idx = degree_profiles.index.get_loc(cs_idx[0])
    degree_name = degree_profiles.iloc[idx]['degree_name']
    print(f'=== Hybrid Top-{K} for {degree_name} (cs) ===')
    display(top_k_jobs_hybrid(idx, K))
else:
    print('No CS degree found — showing first degree profile instead.')
    display(top_k_jobs_hybrid(0, K))

In [ ]:
# For Section 8:
HYBRID_MATRIX_CACHE = CACHE_DIR / f'hybrid_matrix_{JOB_CORPUS_VERSION}_{SEMANTIC_RETRIEVAL_VERSION}_{SKILL_SIGNAL_VERSION}.npy'
np.save(HYBRID_MATRIX_CACHE, hybrid_matrix)
print(f'Saved hybrid matrix to: {HYBRID_MATRIX_CACHE}')


---
## Section 6: Job Clustering (Scalability Layer)

Clustering the filtered, deduplicated in-scope job ads into role-level groups reduces sensitivity to firm-specific wording and provides MOE with a cleaner **degree ↔ job role category** view.


In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.feature_extraction.text import TfidfVectorizer

N_CLUSTERS = 75  # adjust based on corpus size; 50–100 is a sensible range
CLUSTER_CACHE = CACHE_DIR / f'job_clusters_k{N_CLUSTERS}_{JOB_CORPUS_VERSION}.npy'

rebuild_clusters = True
if CLUSTER_CACHE.exists():
    cached_cluster_labels = np.load(CLUSTER_CACHE)
    if cached_cluster_labels.shape[0] == len(jobs):
        print('Loading cluster labels from cache...')
        cluster_labels = cached_cluster_labels
        rebuild_clusters = False
    else:
        print('Cached cluster labels do not match the current filtered job corpus; rebuilding...')

if rebuild_clusters:
    print(f'Clustering {len(jobs):,} jobs into {N_CLUSTERS} clusters...')
    t0 = time.time()
    kmeans = MiniBatchKMeans(
        n_clusters=N_CLUSTERS,
        random_state=42,
        batch_size=2048,
        n_init=5,
        max_iter=300,
    )
    cluster_labels = kmeans.fit_predict(job_embeddings)
    np.save(CLUSTER_CACHE, cluster_labels)
    print(f'Done in {time.time()-t0:.1f}s — inertia: {kmeans.inertia_:.1f}')

jobs['cluster_id'] = cluster_labels
print('Cluster size distribution:')
print(jobs['cluster_id'].value_counts().describe().round(0))


In [ ]:
# Label each cluster with its top TF-IDF terms and most common MCF category
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words='english', min_df=2)
tfidf.fit(jobs['description_clean'].tolist())
feature_names = np.array(tfidf.get_feature_names_out())

cluster_info = []
for c in range(N_CLUSTERS):
    mask = jobs['cluster_id'] == c
    cluster_jobs = jobs[mask]
    n = mask.sum()

    # Top TF-IDF terms
    vecs = tfidf.transform(cluster_jobs['description_clean'].tolist())
    mean_tfidf = np.asarray(vecs.mean(axis=0)).flatten()
    top_term_idx = mean_tfidf.argsort()[::-1][:8]
    top_terms = ', '.join(feature_names[top_term_idx])

    # Most common MCF category
    all_cats = [c for cats in cluster_jobs['categories'].tolist() for c in (cats if isinstance(cats, list) else [])]
    top_cat = pd.Series(all_cats).value_counts().index[0] if all_cats else 'Unknown'

    # Most common title words
    title_words = pd.Series(' '.join(cluster_jobs['title'].tolist()).lower().split()).value_counts()
    common_title = ', '.join(title_words.head(5).index.tolist())

    cluster_info.append({
        'cluster_id'   : c,
        'n_jobs'       : n,
        'top_category' : top_cat,
        'top_terms'    : top_terms,
        'common_titles': common_title,
    })

cluster_df = pd.DataFrame(cluster_info).sort_values('n_jobs', ascending=False)
print('Top 20 clusters by size:')
display(cluster_df.head(20))

In [ ]:
# Compute degree-to-cluster alignment matrix using cluster centroids
# Centroid = mean of unit-norm embeddings → re-normalise for cosine sim

cluster_centroids = np.zeros((N_CLUSTERS, job_embeddings.shape[1]), dtype=np.float32)
for c in range(N_CLUSTERS):
    mask = cluster_labels == c
    if mask.sum() > 0:
        centroid = job_embeddings[mask].mean(axis=0)
        norm = np.linalg.norm(centroid)
        cluster_centroids[c] = centroid / norm if norm > 0 else centroid

# degree-cluster similarity: (n_degrees × n_clusters)
module_cluster_sim = module_embeddings @ cluster_centroids.T
degree_cluster_sim = np.zeros((len(degree_profiles), N_CLUSTERS), dtype=np.float32)

for d_idx, degree_id in enumerate(degree_profiles['degree_id'].tolist()):
    module_idx = degree_module_indices.get(degree_id, np.array([], dtype=int))
    degree_cluster_sim[d_idx] = aggregate_top_k_similarity(module_cluster_sim[module_idx], DEGREE_MODULE_TOP_K)

print(f'Degree-cluster similarity matrix: {degree_cluster_sim.shape}')

In [ ]:
# Build a labelled degree-cluster alignment table
cluster_label_map = dict(zip(cluster_df['cluster_id'], cluster_df['top_category']))
cluster_terms_map = dict(zip(cluster_df['cluster_id'], cluster_df['top_terms']))

alignment_rows = []

for d_idx, d_row in degree_profiles.iterrows():
    d_pos = degree_profiles.index.get_loc(d_idx)
    scores = degree_cluster_sim[d_pos]
    top_clusters = np.argsort(scores)[::-1][:5]

    for rank, c_id in enumerate(top_clusters, 1):
        alignment_rows.append({
            'degree_id'      : d_row['degree_id'],
            'degree'         : d_row['degree_name'],
            'rank'           : rank,
            'cluster_id'     : int(c_id),
            'cluster_label'  : cluster_label_map.get(int(c_id), f'Cluster {c_id}'),
            'cluster_terms'  : cluster_terms_map.get(int(c_id), ''),
            'sim_score'      : round(float(scores[c_id]), 4),
        })


alignment_table = pd.DataFrame(alignment_rows)
alignment_table.to_csv(CACHE_DIR / 'degree_cluster_alignment.csv', index=False)

print('Sample: top-3 cluster matches per degree')
display(alignment_table[alignment_table['rank'] <= 3])

In [ ]:
# For Section 8:
CLUSTER_DF_CACHE = CACHE_DIR / f'cluster_df_{JOB_CORPUS_VERSION}.csv'
DEGREE_CLUSTER_SIM_CACHE = CACHE_DIR / f'degree_cluster_sim_{JOB_CORPUS_VERSION}_{SEMANTIC_RETRIEVAL_VERSION}.npy'

cluster_df.to_csv(CLUSTER_DF_CACHE, index=False)
np.save(DEGREE_CLUSTER_SIM_CACHE, degree_cluster_sim)
print(f'Saved cluster labels to: {CLUSTER_DF_CACHE}')
print(f'Saved degree-cluster similarity to: {DEGREE_CLUSTER_SIM_CACHE}')


---
## Section 7: Evaluation Framework

We construct a stratified golden test set of degree-job pairs, annotate them with human judgement labels, then compute retrieval quality metrics.

### 7a - Golden Test Set Construction

In [ ]:
if Path.cwd().name == 'notebooks':
    EVAL_DIR = Path('evaluation')
else:
    EVAL_DIR = Path('notebooks/evaluation')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

TOP_K_FOR_ANNOTATION = 10
GOLDEN_TOP10_PATH = EVAL_DIR / 'golden_top10_jobs_per_degree.csv'

annotation_rows = []

for d_idx, d_row in degree_profiles.reset_index(drop=True).iterrows():
    scores = hybrid_matrix[d_idx]
    top_job_idx = np.argsort(scores)[::-1][:TOP_K_FOR_ANNOTATION]

    for rank, j_idx in enumerate(top_job_idx, start=1):
        job_row = jobs.iloc[j_idx]

        annotation_rows.append({
            'degree_id': d_row['degree_id'],
            'degree_name': d_row['degree_name'],
            'rank': rank,
            'job_id': job_row['job_id'],
            'job_title': job_row['title'],
            'company': job_row.get('company', ''),
            'job_categories': job_row.get('categories_str', ''),
            'job_description_snippet': str(job_row.get('description_clean', ''))[:600],
            'cosine_sim': round(float(sim_matrix[d_idx, j_idx]), 4),
            'skill_coverage': round(float(skill_coverage_matrix[d_idx, j_idx]), 4),
            'hybrid_score': round(float(hybrid_matrix[d_idx, j_idx]), 4),
            'human_label': '',          # fill: 2=Relevant, 1=Somewhat Relevant, 0=Not Relevant
            'annotator_notes': '',
        })

golden_top10_df = pd.DataFrame(annotation_rows)
golden_top10_df.to_csv(GOLDEN_TOP10_PATH, index=False)

print(f'Saved top-{TOP_K_FOR_ANNOTATION} jobs per degree to: {GOLDEN_TOP10_PATH}')
print(f'Total annotation rows: {len(golden_top10_df):,}')
display(golden_top10_df.head(20))

In [ ]:
print(GOLDEN_TOP10_PATH.resolve())

### 7b - Evaluation Metrics

Run this cell after filling in the `human_label` column in `evaluation/golden_top10_jobs_per_degree.csv`.

In [ ]:
from pathlib import Path
from scipy.stats import spearmanr

if Path.cwd().name == 'notebooks':
    GOLDEN_TOP10_PATH = Path('evaluation/golden_top10_jobs_per_degree.csv')
else:
    GOLDEN_TOP10_PATH = Path('notebooks/evaluation/golden_top10_jobs_per_degree.csv')

if 'golden_top10_df' not in globals():
    golden_top10_df = pd.read_csv(GOLDEN_TOP10_PATH)
    print(f'Loaded annotation file from: {GOLDEN_TOP10_PATH}')


def precision_at_k(labels: list[int], k: int) -> float:
    """Fraction of top-k retrieved jobs that are relevant or somewhat relevant."""
    top_k = labels[:k]
    if not top_k:
        return 0.0
    return sum(label >= 1 for label in top_k) / len(top_k)


def top_k_hit_rate(labels: list[int], k: int) -> int:
    """1 if at least one Relevant job appears in top-k, else 0."""
    return int(any(label == 2 for label in labels[:k]))


def ndcg_at_k(labels: list[int], k: int) -> float:
    """Normalised Discounted Cumulative Gain at rank k."""
    labels = labels[:k]

    def dcg(values: list[int]) -> float:
        return sum((2**label - 1) / np.log2(i + 2) for i, label in enumerate(values))

    ideal = sorted(labels, reverse=True)
    ideal_dcg = dcg(ideal)
    return dcg(labels) / ideal_dcg if ideal_dcg > 0 else 0.0


def evaluate_top10_annotations(golden: pd.DataFrame, k_precision: int = 5, k_hit: int = 3):
    """
    Evaluate Section 7a's top-10 annotation CSV.
    human_label: 2=Relevant, 1=Somewhat Relevant, 0=Not Relevant.
    """
    annotated = golden[
        golden['human_label'].apply(lambda x: str(x).strip()).isin(['0', '1', '2'])
    ].copy()

    annotated['human_label'] = annotated['human_label'].astype(int)

    if annotated.empty:
        print('No annotated pairs found. Fill in human_label first.')
        return None

    results = []

    for degree_id, grp in annotated.groupby('degree_id'):
        grp_sorted = grp.sort_values('rank')
        labels_ranked = grp_sorted['human_label'].tolist()

        degree_name = grp_sorted['degree_name'].iloc[0]

        results.append({
            'degree_id': degree_id,
            'degree_name': degree_name,
            f'Precision@{k_precision}': precision_at_k(labels_ranked, k_precision),
            f'NDCG@{k_precision}': ndcg_at_k(labels_ranked, k_precision),
            f'Top-{k_hit}_HitRate': top_k_hit_rate(labels_ranked, k_hit),
            'annotated_jobs': len(grp_sorted),
        })

    metrics_df = pd.DataFrame(results)

    rho, pval = spearmanr(annotated['hybrid_score'], annotated['human_label'])

    print('=== Evaluation Results ===')
    print(f'Annotated pairs   : {len(annotated)}')
    print(f'Degrees evaluated : {annotated["degree_id"].nunique()}')
    print(f'Spearman rho      : {rho:.4f}  (p={pval:.4f})')
    print()
    display(metrics_df)
    print()
    display(metrics_df.describe(numeric_only=True).round(3))

    return metrics_df, rho, pval


eval_results = evaluate_top10_annotations(golden_top10_df)

### 7c - Coverage Report

In [ ]:
from pathlib import Path

# If the notebook is running from notebooks/, use local folders.
# If it is running from repo root, use notebooks/cache and notebooks/evaluation.
if Path.cwd().name == 'notebooks':
    CACHE_DIR = Path('cache')
    EVAL_DIR = Path('evaluation')
else:
    CACHE_DIR = Path('notebooks/cache')
    EVAL_DIR = Path('notebooks/evaluation')

JOBS_RAW_CACHE = CACHE_DIR / 'jobs_raw.parquet'
JOBS_CLEAN_CACHE = CACHE_DIR / f'jobs_clean_{JOB_CORPUS_VERSION}.parquet'
JOB_SCOPE_CACHE = CACHE_DIR / f'jobs_scope_audit_{JOB_CORPUS_VERSION}.parquet'
DEGREE_META_CACHE = CACHE_DIR / 'degree_profiles.parquet'
GOLDEN_TOP10_PATH = EVAL_DIR / 'golden_top10_jobs_per_degree.csv'

# Reload cached outputs if this section is run after a kernel restart.
if 'jobs_raw' not in globals() and JOBS_RAW_CACHE.exists():
    jobs_raw = pd.read_parquet(JOBS_RAW_CACHE)

if 'jobs' not in globals() and JOBS_CLEAN_CACHE.exists():
    jobs = pd.read_parquet(JOBS_CLEAN_CACHE)

if 'jobs_scope_audit' not in globals() and JOB_SCOPE_CACHE.exists():
    jobs_scope_audit = pd.read_parquet(JOB_SCOPE_CACHE)

if 'degree_profiles' not in globals() and DEGREE_META_CACHE.exists():
    degree_profiles = pd.read_parquet(DEGREE_META_CACHE)

if 'golden_top10_df' not in globals() and GOLDEN_TOP10_PATH.exists():
    golden_top10_df = pd.read_csv(GOLDEN_TOP10_PATH)

print('=' * 50)
print('  FRAMEWORK COVERAGE REPORT')
print('=' * 50)

# Job ad coverage
print('\n[Job Ads]')
if 'job_files' in globals():
    print(f'  Files found           : {len(job_files):>8,}')
else:
    print('  Files found           : unavailable; run Section 1 job loading first')

if 'jobs_raw' in globals() and 'job_files' in globals() and len(job_files) > 0:
    print(f'  Successfully parsed   : {len(jobs_raw):>8,}  ({len(jobs_raw)/len(job_files)*100:.1f}%)')
elif 'jobs_raw' in globals():
    print(f'  Successfully parsed   : {len(jobs_raw):>8,}')
else:
    print('  Successfully parsed   : unavailable')

if 'jobs_scope_audit' in globals() and 'job_files' in globals() and len(job_files) > 0:
    print(f'  Clean after text      : {len(jobs_scope_audit):>8,}  ({len(jobs_scope_audit)/len(job_files)*100:.1f}%)')
    excluded = (~jobs_scope_audit['is_target_role']).sum()
    in_scope_before_exact = jobs_scope_audit['is_target_role'].sum()
    exact_removed = (jobs_scope_audit['is_target_role'] & jobs_scope_audit['duplicate_rank'].gt(1)).sum()
    semantic_removed = jobs_scope_audit['semantic_duplicate_rank'].fillna(0).gt(1).sum()
    after_exact = in_scope_before_exact - exact_removed
    print(f'  Out-of-scope removed  : {excluded:>8,}  ({excluded/len(job_files)*100:.1f}%)')
    print(f'  In-scope pre-exact    : {in_scope_before_exact:>8,}  ({in_scope_before_exact/len(job_files)*100:.1f}%)')
    print(f'  Exact repost removed  : {exact_removed:>8,}  ({exact_removed/len(job_files)*100:.1f}%)')
    print(f'  After exact dedup     : {after_exact:>8,}  ({after_exact/len(job_files)*100:.1f}%)')
    print(f'  Semantic near-dupes   : {semantic_removed:>8,}  ({semantic_removed/len(job_files)*100:.1f}%)')
elif 'jobs_scope_audit' in globals():
    print(f'  Clean after text      : {len(jobs_scope_audit):>8,}')

if 'jobs' in globals() and 'job_files' in globals() and len(job_files) > 0:
    print(f'  Final retrieval corpus: {len(jobs):>8,}  ({len(jobs)/len(job_files)*100:.1f}%)')
elif 'jobs' in globals():
    print(f'  Final retrieval corpus: {len(jobs):>8,}')
else:
    print('  Final retrieval corpus: unavailable')

if 'jobs' in globals() and 'skills' in jobs.columns:
    skill_count = jobs['skills'].apply(len).gt(0).sum()
    skill_pct = jobs['skills'].apply(len).gt(0).mean() * 100
    print(f'  With skill tags       : {skill_count:>8,}  ({skill_pct:.1f}%)')

if 'jobs_scope_audit' in globals():
    scope_counts = jobs_scope_audit.loc[~jobs_scope_audit['is_target_role'], 'role_scope'].value_counts()
    if not scope_counts.empty:
        print('  Scope exclusions      :')
        for reason, count in scope_counts.items():
            print(f'    - {reason:<24} {count:>8,}')

# Module coverage
print('\n[NUS Modules]')
if 'modules_raw' in globals():
    print(f'  Total rows in CSV     : {len(modules_raw):>8,}')
else:
    print('  Total rows in CSV     : unavailable; run Section 1 module loading first')

if 'modules_clean' in globals() and 'modules_raw' in globals() and len(modules_raw) > 0:
    print(f'  With usable desc      : {len(modules_clean):>8,}  ({len(modules_clean)/len(modules_raw)*100:.1f}%)')
    if 'is_undergrad' in modules_clean.columns:
        print(f'  Undergrad (1-4xxx)    : {modules_clean["is_undergrad"].sum():>8,}  ({modules_clean["is_undergrad"].mean()*100:.1f}%)')
elif 'modules_clean' in globals():
    print(f'  With usable desc      : {len(modules_clean):>8,}')
else:
    print('  With usable desc      : unavailable')

# Degree profile coverage
print('\n[Degree Profiles]')
if 'degree_profiles' in globals():
    print(f'  Degrees profiled      : {len(degree_profiles):>8,}')
    if 'n_modules' in degree_profiles.columns:
        print(f'  Avg modules/degree    : {degree_profiles["n_modules"].mean():>8.1f}')
        print(f'  Min modules/degree    : {degree_profiles["n_modules"].min():>8,}')
        print(f'  Max modules/degree    : {degree_profiles["n_modules"].max():>8,}')
else:
    print('  Degrees profiled      : unavailable; run Section 2 first')

# Alignment coverage
print('\n[Alignment Coverage]')
if 'degree_profiles' in globals() and 'jobs' in globals():
    total_pairs = len(degree_profiles) * len(jobs)
    print(f'  Degrees x Jobs pairs  : {len(degree_profiles):,} x {len(jobs):,} = {total_pairs:,}')

if 'sim_matrix' in globals():
    print(f'  Baseline matrix shape : {sim_matrix.shape}')

if 'skill_coverage_matrix' in globals():
    nonzero_skill = (skill_coverage_matrix > 0).sum()
    total_skill = skill_coverage_matrix.size
    print(f'  Skill coverage nonzero: {nonzero_skill:,} / {total_skill:,}  ({nonzero_skill/total_skill*100:.2f}%)')

if 'hybrid_matrix' in globals():
    print(f'  Hybrid matrix shape   : {hybrid_matrix.shape}')

# Section 7a annotation set coverage
print('\n[Golden Annotation Set]')
if 'golden_top10_df' in globals():
    print(f'  Annotation rows       : {len(golden_top10_df):>8,}')
    print(f'  Degrees represented   : {golden_top10_df["degree_id"].nunique():>8,}')
    print(f'  Jobs per degree       : {golden_top10_df.groupby("degree_id").size().min()}-{golden_top10_df.groupby("degree_id").size().max()}')
    labelled = golden_top10_df['human_label'].apply(lambda x: str(x).strip()).isin(['0', '1', '2']).sum()
    print(f'  Human-labelled rows   : {labelled:>8,}  ({labelled/len(golden_top10_df)*100:.1f}%)')
else:
    print('  Annotation CSV        : unavailable; run Section 7a first')

print('=' * 50)


---
## Section 8: Results Summary & Interpretation

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

# Use the correct local folders whether the notebook cwd is repo root or notebooks/.
if Path.cwd().name == 'notebooks':
    CACHE_DIR = Path('cache')
else:
    CACHE_DIR = Path('notebooks/cache')

DEGREE_META_CACHE = CACHE_DIR / 'degree_profiles.parquet'
DEGREE_CLUSTER_SIM_CACHE = CACHE_DIR / f'degree_cluster_sim_{JOB_CORPUS_VERSION}_{SEMANTIC_RETRIEVAL_VERSION}.npy'
CLUSTER_DF_CACHE = CACHE_DIR / f'cluster_df_{JOB_CORPUS_VERSION}.csv'

# Reload required Section 2 / Section 6 outputs if this section is run after a kernel restart.
if 'degree_profiles' not in globals():
    if DEGREE_META_CACHE.exists():
        degree_profiles = pd.read_parquet(DEGREE_META_CACHE)
        print(f'Loaded degree profiles from: {DEGREE_META_CACHE}')
    else:
        raise FileNotFoundError('Run Section 2 first to build degree_profiles.')

if 'degree_cluster_sim' not in globals():
    if DEGREE_CLUSTER_SIM_CACHE.exists():
        degree_cluster_sim = np.load(DEGREE_CLUSTER_SIM_CACHE)
        print(f'Loaded degree-cluster similarity from: {DEGREE_CLUSTER_SIM_CACHE}')
    else:
        raise FileNotFoundError('Run Section 6 first to build degree_cluster_sim.')

if 'cluster_df' not in globals():
    if CLUSTER_DF_CACHE.exists():
        cluster_df = pd.read_csv(CLUSTER_DF_CACHE)
        print(f'Loaded cluster labels from: {CLUSTER_DF_CACHE}')
    else:
        raise FileNotFoundError('Run Section 6 first to build cluster_df.')

if degree_cluster_sim.shape[0] != len(degree_profiles):
    raise ValueError('Cached degree_cluster_sim does not match current degree_profiles. Re-run Section 6.')

# Select top N degrees and top M clusters for the heatmap
TOP_DEG = min(5, len(degree_profiles))
TOP_CLU = min(20, len(cluster_df))

# Pick the degrees with the highest max cluster alignment
max_cluster_score = degree_cluster_sim.max(axis=1)
top_deg_idx = np.argsort(max_cluster_score)[::-1][:TOP_DEG]

if 'degree_id' in degree_profiles.columns:
    top_deg_labels = degree_profiles.iloc[top_deg_idx]['degree_id'].tolist()
else:
    top_deg_labels = degree_profiles.iloc[top_deg_idx]['degree_name'].tolist()

# Pick the top clusters by size
cluster_df_sorted = cluster_df.sort_values('n_jobs', ascending=False).reset_index(drop=True)
top_clu_ids = cluster_df_sorted.head(TOP_CLU)['cluster_id'].astype(int).tolist()
top_clu_labels = cluster_df_sorted.head(TOP_CLU)['top_category'].astype(str).tolist()

heatmap_data = degree_cluster_sim[np.ix_(top_deg_idx, top_clu_ids)]

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(heatmap_data, aspect='auto', cmap='YlOrRd')
plt.colorbar(im, ax=ax, label='Cosine Similarity')

ax.set_xticks(range(TOP_CLU))
ax.set_xticklabels(top_clu_labels, rotation=45, ha='right', fontsize=8)

ax.set_yticks(range(TOP_DEG))
ax.set_yticklabels(top_deg_labels, fontsize=9)

ax.set_title('Degree-Job Role Cluster Alignment Heatmap', fontsize=13)
ax.set_xlabel('Job Role Cluster (by MCF Category)', fontsize=10)
ax.set_ylabel('NUS Degree Programme', fontsize=10)

plt.tight_layout()
plt.savefig(CACHE_DIR / 'alignment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from pathlib import Path

# Use the correct local folder whether the notebook cwd is repo root or notebooks/.
if Path.cwd().name == 'notebooks':
    CACHE_DIR = Path('cache')
else:
    CACHE_DIR = Path('notebooks/cache')

DEGREE_META_CACHE = CACHE_DIR / 'degree_profiles.parquet'
JOBS_CLEAN_CACHE = CACHE_DIR / f'jobs_clean_{JOB_CORPUS_VERSION}.parquet'
HYBRID_MATRIX_CACHE = CACHE_DIR / f'hybrid_matrix_{JOB_CORPUS_VERSION}_{SEMANTIC_RETRIEVAL_VERSION}_{SKILL_SIGNAL_VERSION}.npy'
SIM_MATRIX_CACHE = CACHE_DIR / f'sim_matrix_{MODEL_NAME.replace("/","_")}_{JOB_CORPUS_VERSION}_{SEMANTIC_RETRIEVAL_VERSION}.npy'
SKILL_COVERAGE_CACHE = CACHE_DIR / f'skill_coverage_matrix_{JOB_CORPUS_VERSION}_{SKILL_SIGNAL_VERSION}.npy'

# Reload earlier section outputs if this cell is run after a kernel restart.
if 'degree_profiles' not in globals():
    degree_profiles = pd.read_parquet(DEGREE_META_CACHE)
    print(f'Loaded degree profiles from: {DEGREE_META_CACHE}')

if 'jobs' not in globals():
    jobs = pd.read_parquet(JOBS_CLEAN_CACHE)
    print(f'Loaded clean jobs from: {JOBS_CLEAN_CACHE}')

expected_matrix_shape = (len(degree_profiles), len(jobs))

if 'hybrid_matrix' not in globals():
    hybrid_matrix = np.load(HYBRID_MATRIX_CACHE)
    if hybrid_matrix.shape != expected_matrix_shape:
        raise ValueError('Cached hybrid_matrix does not match the current degree/job corpus. Re-run Section 5.')
    print(f'Loaded hybrid matrix from: {HYBRID_MATRIX_CACHE}')

if 'sim_matrix' not in globals() and SIM_MATRIX_CACHE.exists():
    sim_matrix = np.load(SIM_MATRIX_CACHE)
    if sim_matrix.shape != expected_matrix_shape:
        raise ValueError('Cached sim_matrix does not match the current degree/job corpus. Re-run Sections 3-4.')
    print(f'Loaded similarity matrix from: {SIM_MATRIX_CACHE}')

if 'skill_coverage_matrix' not in globals() and SKILL_COVERAGE_CACHE.exists():
    skill_coverage_matrix = np.load(SKILL_COVERAGE_CACHE)
    if skill_coverage_matrix.shape != expected_matrix_shape:
        raise ValueError('Cached skill_coverage_matrix does not match the current degree/job corpus. Re-run Section 5.')
    print(f'Loaded skill coverage matrix from: {SKILL_COVERAGE_CACHE}')

# Top-5 matching job roles by hybrid score per NUS degree
TOP_JOBS_PER_DEGREE = 5
SUMMARY_PATH = CACHE_DIR / 'degree_job_alignment_summary.csv'

summary_rows = []

for d_pos, d_row in degree_profiles.reset_index(drop=True).iterrows():
    scores = hybrid_matrix[d_pos]
    top_idx = np.argsort(scores)[::-1][:TOP_JOBS_PER_DEGREE]

    for rank, j_idx in enumerate(top_idx, start=1):
        job_row = jobs.iloc[j_idx]

        row = {
            'degree_id': d_row['degree_id'],
            'degree_name': d_row['degree_name'],
            'rank': rank,
            'job_id': job_row['job_id'],
            'job_title': job_row['title'],
            'company': job_row.get('company', ''),
            'job_categories': job_row.get('categories_str', ''),
            'hybrid_score': round(float(hybrid_matrix[d_pos, j_idx]), 4),
        }

        if 'sim_matrix' in globals():
            row['cosine_sim'] = round(float(sim_matrix[d_pos, j_idx]), 4)

        if 'skill_coverage_matrix' in globals():
            row['skill_coverage'] = round(float(skill_coverage_matrix[d_pos, j_idx]), 4)

        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_PATH, index=False)

print(f'Summary table saved to: {SUMMARY_PATH}')
print(f'Total rows: {len(summary_df):,}')
display(summary_df.head(20))


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

# Use the correct local folder whether the notebook cwd is repo root or notebooks/.
if Path.cwd().name == 'notebooks':
    CACHE_DIR = Path('cache')
else:
    CACHE_DIR = Path('notebooks/cache')

DEGREE_META_CACHE = CACHE_DIR / 'degree_profiles.parquet'
HYBRID_MATRIX_CACHE = CACHE_DIR / f'hybrid_matrix_{JOB_CORPUS_VERSION}_{SEMANTIC_RETRIEVAL_VERSION}_{SKILL_SIGNAL_VERSION}.npy'

# Reload earlier section outputs if this cell is run after a kernel restart.
if 'degree_profiles' not in globals():
    if DEGREE_META_CACHE.exists():
        degree_profiles = pd.read_parquet(DEGREE_META_CACHE)
        print(f'Loaded degree profiles from: {DEGREE_META_CACHE}')
    else:
        raise FileNotFoundError('Run Section 2 first to build degree_profiles.')

if 'hybrid_matrix' not in globals():
    if HYBRID_MATRIX_CACHE.exists():
        hybrid_matrix = np.load(HYBRID_MATRIX_CACHE)
        print(f'Loaded hybrid matrix from: {HYBRID_MATRIX_CACHE}')
    else:
        raise FileNotFoundError('Run Section 5 first to build hybrid_matrix.')

if hybrid_matrix.shape[0] != len(degree_profiles):
    raise ValueError('Cached hybrid_matrix does not match current degree_profiles. Re-run Section 5.')

# Degree-level alignment score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of top-1 hybrid score per degree
top1_scores = hybrid_matrix.max(axis=1)
axes[0].hist(top1_scores, bins=min(10, len(top1_scores)), color='mediumseagreen', edgecolor='white')
axes[0].axvline(top1_scores.mean(), color='red', linestyle='--', label=f'Mean={top1_scores.mean():.3f}')
axes[0].set_title('Best Job Match Score per Degree\n(Hybrid Score)')
axes[0].set_xlabel('Hybrid Alignment Score')
axes[0].set_ylabel('Number of Degrees')
axes[0].legend()

# Top degrees by max alignment
degree_profiles_plot = degree_profiles.copy()
degree_profiles_plot['max_hybrid_score'] = top1_scores

top_aligned = degree_profiles_plot.nlargest(
    min(5, len(degree_profiles_plot)),
    'max_hybrid_score'
)[['degree_name', 'degree_id', 'max_hybrid_score']]

labels = [
    f'{name} ({deg_id})'
    for name, deg_id in zip(top_aligned['degree_name'], top_aligned['degree_id'])
]

axes[1].barh(labels, top_aligned['max_hybrid_score'], color='steelblue')
axes[1].set_xlabel('Max Hybrid Alignment Score')
axes[1].set_title('Top Degrees by Best Job Match')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(CACHE_DIR / 'degree_alignment_scores.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from pathlib import Path

# Use the correct local folders whether the notebook cwd is repo root or notebooks/.
if Path.cwd().name == 'notebooks':
    CACHE_DIR = Path('cache')
    EVAL_DIR = Path('evaluation')
else:
    CACHE_DIR = Path('notebooks/cache')
    EVAL_DIR = Path('notebooks/evaluation')

JOBS_CLEAN_CACHE = CACHE_DIR / f'jobs_clean_{JOB_CORPUS_VERSION}.parquet'
JOB_SCOPE_CACHE = CACHE_DIR / f'jobs_scope_audit_{JOB_CORPUS_VERSION}.parquet'
DEGREE_META_CACHE = CACHE_DIR / 'degree_profiles.parquet'
GOLDEN_TOP10_PATH = EVAL_DIR / 'golden_top10_jobs_per_degree.csv'

SUMMARY_PATH = CACHE_DIR / 'degree_job_alignment_summary.csv'
CLUSTER_ALIGNMENT_PATH = CACHE_DIR / 'degree_cluster_alignment.csv'
HEATMAP_PATH = CACHE_DIR / 'alignment_heatmap.png'
DEGREE_SCORE_PLOT_PATH = CACHE_DIR / 'degree_alignment_scores.png'

# Reload earlier section outputs if this cell is run after a kernel restart.
if 'jobs' not in globals() and JOBS_CLEAN_CACHE.exists():
    jobs = pd.read_parquet(JOBS_CLEAN_CACHE)

if 'jobs_scope_audit' not in globals() and JOB_SCOPE_CACHE.exists():
    jobs_scope_audit = pd.read_parquet(JOB_SCOPE_CACHE)

if 'degree_profiles' not in globals() and DEGREE_META_CACHE.exists():
    degree_profiles = pd.read_parquet(DEGREE_META_CACHE)

if 'golden_top10_df' not in globals() and GOLDEN_TOP10_PATH.exists():
    golden_top10_df = pd.read_csv(GOLDEN_TOP10_PATH)

model_name = globals().get('MODEL_NAME', 'all-MiniLM-L6-v2')
semantic_method = globals().get('SEMANTIC_RETRIEVAL_VERSION', 'not set')
module_top_k = globals().get('DEGREE_MODULE_TOP_K', 'not set')
job_desc_limit = globals().get('JOB_DESC_WORD_LIMIT', 'not set')
alpha = globals().get('ALPHA', 'not set')
beta = globals().get('BETA', 'not set')
n_clusters = globals().get('N_CLUSTERS', 'not run')

print('=' * 60)
print('  NUS CURRICULUM-JOB ALIGNMENT FRAMEWORK - SUMMARY')
print('=' * 60)
print(f'\nModel        : {model_name}')
print(f'Job corpus   : {JOB_CORPUS_VERSION}')
print(f'Semantics    : {semantic_method}')
print(f'Module top-k : {module_top_k}')
print(f'Job desc cap : {job_desc_limit} words')
print(f'Hybrid alpha : {alpha} (semantic)')
print(f'Hybrid beta  : {beta} (job-skill coverage)')
print(f'Job clusters : {n_clusters}')

print('\nScale')
if 'jobs' in globals():
    print(f'  Job ads analyzed       : {len(jobs):,}')
else:
    print('  Job ads analyzed       : unavailable; run Section 1')

if 'jobs_scope_audit' in globals():
    excluded = (~jobs_scope_audit['is_target_role']).sum()
    exact_removed = (jobs_scope_audit['is_target_role'] & jobs_scope_audit['duplicate_rank'].gt(1)).sum()
    semantic_removed = jobs_scope_audit['semantic_duplicate_rank'].fillna(0).gt(1).sum()
    print(f'  Out-of-scope removed   : {excluded:,}')
    print(f'  Exact repost removed   : {exact_removed:,}')
    print(f'  Semantic near-dupes    : {semantic_removed:,}')

if 'degree_profiles' in globals():
    print(f'  Degree profiles        : {len(degree_profiles):,}')
else:
    print('  Degree profiles        : unavailable; run Section 2')

if 'jobs' in globals() and 'degree_profiles' in globals():
    print(f'  Degree-job pairs scored: {len(degree_profiles) * len(jobs):,}')
else:
    print('  Degree-job pairs scored: unavailable; run Sections 2 and 5')

print('\nKey Outputs')
print(f'  {SUMMARY_PATH} - top-5 jobs per degree')
print(f'  {CLUSTER_ALIGNMENT_PATH} - top-5 job clusters per degree')
print(f'  {HEATMAP_PATH} - degree x job cluster heatmap')
print(f'  {DEGREE_SCORE_PLOT_PATH} - score distribution')

print('\nGolden Set for Human Annotation')
print(f'  {GOLDEN_TOP10_PATH}')
if 'golden_top10_df' in globals():
    labelled = golden_top10_df['human_label'].apply(lambda x: str(x).strip()).isin(['0', '1', '2']).sum()
    print(f'  Rows                  : {len(golden_top10_df):,}')
    print(f'  Degrees represented   : {golden_top10_df["degree_id"].nunique():,}')
    print(f'  Human-labelled rows   : {labelled:,} / {len(golden_top10_df):,}')
print('  Fill human_label as 2=Relevant, 1=Somewhat Relevant, 0=Not Relevant')
print('  Then re-run Section 7b for Precision@K, NDCG@K, Top-3 hit rate, and Spearman rho')

print('=' * 60)
